# LABORATORIO N.° 07 — Causalidad: correlación no es causa
## Evaluación causal de una intervención real con Diferencias en Diferencias

**Curso:** Analítica Empresarial Integrada  
**Semana:** 7  
**Docente:** Pilar Rocío Sayán Mejía  
**Fuente:** UCI Machine Learning Repository — Online Retail II (ID 502)  
**Evento:** aumento de la tasa estándar de VAT del Reino Unido, 17.5 % → 20 %, vigente desde el **04/01/2011**.

> **Pregunta orientadora:** ¿Cambió el desempeño comercial del mercado británico después del evento más de lo que cambió un grupo internacional de comparación?

**Advertencia metodológica:** este notebook es un ejercicio de inferencia causal observacional. La base no registra la tasa de VAT aplicada por transacción ni garantiza que los mercados internacionales sean controles no expuestos. La interpretación causal depende de supuestos que deberán ser auditados.

**Horario de práctica:** 8:30–10:10 (100 minutos, con receso de 9:15 a 9:20).



## Agenda de la práctica

| Hora | Actividad |
|---|---|
| 8:30–8:40 | Apertura: caso, intervención y pregunta contrafactual |
| 8:40–8:50 | Fuente UCI, trazabilidad y controles de calidad |
| 8:50–9:05 | Grupos, ventana y variable de resultado |
| 9:05–9:15 | DAG, confusores y validez del control |
| 9:15–9:20 | Receso |
| 9:20–9:35 | Tendencias previas y supuesto de paralelismo |
| 9:35–9:50 | DiD manual, regresión e interpretación |
| 9:50–10:10 | Robustez, conclusión y ticket de salida |


## 0. Antes de calcular: qué identifica DiD

DiD compara **cambios**:

\[
DiD=(Y_{T,post}-Y_{T,pre})-(Y_{C,post}-Y_{C,pre})
\]

La estimación solo admite una lectura causal defendible si el grupo de comparación aproxima razonablemente el contrafactual del grupo tratado. El supuesto central es el de **tendencias paralelas** en ausencia de la intervención.


In [ ]:
# CELDA 1 — Entorno reproducible
%pip install -q openpyxl statsmodels

import warnings, io, zipfile, urllib.request, numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
print("Entorno listo.")

Entorno listo.


## 1. Descarga y trazabilidad

Se descarga **Online Retail II** desde UCI. Es una base real de transacciones de un comercio electrónico registrado en Reino Unido, con observaciones entre diciembre de 2009 y diciembre de 2011.

**No se permite reemplazar la fuente por datos simulados si la descarga falla.** En ese caso, documente el error y vuelva a ejecutar con conexión a internet.


In [ ]:
# CELDA 2 — Descarga desde UCI (archivo oficial ZIP; fetch_ucirepo ya no admite este dataset)
URL_UCI = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
req = urllib.request.Request(URL_UCI, headers={"User-Agent": "Mozilla/5.0"})
print("Descargando Online Retail II desde UCI (~45 MB)...")
with urllib.request.urlopen(req, timeout=300) as r:
    zbytes = r.read()
z = zipfile.ZipFile(io.BytesIO(zbytes))
archivo = z.namelist()[0]

# El Excel trae dos hojas: 2009-2010 y 2010-2011. Se leen y se unen (tarda 2-3 min).
hojas = pd.read_excel(z.open(archivo), sheet_name=None, engine="openpyxl")
df = pd.concat(hojas.values(), ignore_index=True)

print("Fuente  : https://archive.ics.uci.edu/dataset/502/online+retail+ii")
print("DOI     : 10.24432/C5CG6D")
print("Archivo :", archivo, "| hojas:", list(hojas.keys()))
print("Filas   :", len(df))
print("Columnas:", list(df.columns))
display(df.head())


Descargando Online Retail II desde UCI (~45 MB)...
Fuente  : https://archive.ics.uci.edu/dataset/502/online+retail+ii
DOI     : 10.24432/C5CG6D
Archivo : online_retail_II.xlsx | hojas: ['Year 2009-2010', 'Year 2010-2011']
Filas   : 1067371
Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
# CELDA 3 — Normalización robusta de nombres
df.columns = [str(c).strip().replace(" ", "") for c in df.columns]
rename = {
    "Invoice":"InvoiceNo", "InvoiceNo":"InvoiceNo",
    "StockCode":"StockCode", "Description":"Description",
    "Quantity":"Quantity", "InvoiceDate":"InvoiceDate",
    "Price":"UnitPrice", "UnitPrice":"UnitPrice",
    "CustomerID":"CustomerID", "CustomerId":"CustomerID",
    "Country":"Country"
}
df = df.rename(columns={c: rename.get(c,c) for c in df.columns})

required = {"InvoiceNo","Quantity","InvoiceDate","UnitPrice","Country"}
faltan = required - set(df.columns)
assert not faltan, f"Faltan columnas esperadas: {faltan}"

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["Country"] = df["Country"].astype(str).str.strip()

print(df[["InvoiceDate","Quantity","UnitPrice","Country"]].describe(include="all"))


                          InvoiceDate      Quantity     UnitPrice  \
count                         1067371  1.067371e+06  1.067371e+06   
unique                            NaN           NaN           NaN   
top                               NaN           NaN           NaN   
freq                              NaN           NaN           NaN   
mean    2011-01-02 21:13:55.394028544  9.938898e+00  4.649388e+00   
min               2009-12-01 07:45:00 -8.099500e+04 -5.359436e+04   
25%               2010-07-09 09:46:00  1.000000e+00  1.250000e+00   
50%               2010-12-07 15:28:00  3.000000e+00  2.100000e+00   
75%               2011-07-22 10:23:00  1.000000e+01  4.150000e+00   
max               2011-12-09 12:50:00  8.099500e+04  3.897000e+04   
std                               NaN  1.727058e+02  1.235531e+02   

               Country  
count          1067371  
unique              43  
top     United Kingdom  
freq            981330  
mean               NaN  
min                Na

### Pregunta 1 — Trazabilidad
¿Por qué esta base permite construir un análisis temporal antes/después? ¿Qué información causal importante **no** contiene?

**Respuesta:**

La base permite construir un análisis temporal porque contiene la **fecha de cada transacción**, desde diciembre de 2009 hasta diciembre de 2011, lo que permite comparar el desempeño antes y después del evento del 04/01/2011. Sin embargo, no contiene información sobre la **tasa de VAT aplicada a cada transacción** ni garantiza que los mercados internacionales utilizados como comparación no hayan estado expuestos al evento. Por ello, la base permite realizar el análisis DiD, pero la interpretación causal depende de que se cumplan y auditen supuestos como las **tendencias paralelas**.


In [ ]:
# CELDA 4 — Control de calidad
df["cancelacion"] = df["InvoiceNo"].str.upper().str.startswith("C")
control = pd.Series({
    "filas": len(df),
    "fecha_nula": int(df["InvoiceDate"].isna().sum()),
    "cantidad_nula": int(df["Quantity"].isna().sum()),
    "precio_nulo": int(df["UnitPrice"].isna().sum()),
    "cancelaciones": int(df["cancelacion"].sum()),
    "cantidad_no_positiva": int((df["Quantity"] <= 0).sum()),
    "precio_no_positivo": int((df["UnitPrice"] <= 0).sum()),
})
print(control.to_string())

# Análisis principal: ventas válidas, sin fabricar registros
ventas = df.loc[
    df["InvoiceDate"].notna() &
    (~df["cancelacion"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()
ventas["ingreso"] = ventas["Quantity"] * ventas["UnitPrice"]

print("\nFilas válidas para análisis principal:", len(ventas))
print("Periodo:", ventas["InvoiceDate"].min(), "a", ventas["InvoiceDate"].max())


filas                   1067371
fecha_nula                    0
cantidad_nula                 0
precio_nulo                   0
cancelaciones             19494
cantidad_no_positiva      22950
precio_no_positivo         6207

Filas válidas para análisis principal: 1041670
Periodo: 2009-12-01 07:45:00 a 2011-12-09 12:50:00


## 2. Definición del diseño

- **Evento:** 04/01/2011.
- **Tratado académico:** United Kingdom.
- **Comparación base:** France, Germany, Netherlands, Belgium, Spain e Ireland/EIRE cuando estén disponibles.
- **Resultado:** ingreso semanal neto observado de ventas válidas.
- **Ventana base:** 26 semanas antes y 26 semanas después.

La elección del control **no se acepta por defecto**: se audita con cobertura, composición y tendencias previas.

Las semanas son bloques completos de siete días anclados en el 04/01/2011; así ninguna semana mezcla pre y post.


In [ ]:
# CELDA 5 — Parámetros editables
FECHA_EVENTO = pd.Timestamp("2011-01-04")
SEMANAS_PRE = 26
SEMANAS_POST = 26

CONTROL_CANDIDATOS = ["France","Germany","Netherlands","Belgium","Spain","EIRE","Ireland"]
print("Países más frecuentes:")
display(ventas["Country"].value_counts().head(20))


Países más frecuentes:


,count
Country,
United Kingdom,958501
EIRE,17349
Germany,16694
France,13940
Netherlands,5088
Spain,3719
Switzerland,3136
Belgium,3068
Portugal,2562


In [ ]:
# CELDA 6 — Construcción de grupos y ventana
control_disponible = [c for c in CONTROL_CANDIDATOS if c in set(ventas["Country"])]
print("Control disponible:", control_disponible)

inicio = FECHA_EVENTO - pd.Timedelta(weeks=SEMANAS_PRE)
fin = FECHA_EVENTO + pd.Timedelta(weeks=SEMANAS_POST)

base = ventas.loc[
    ventas["InvoiceDate"].between(inicio, fin, inclusive="left") &
    ventas["Country"].isin(["United Kingdom"] + control_disponible)
].copy()

base["tratado"] = (base["Country"] == "United Kingdom").astype(int)
base["post"] = (base["InvoiceDate"] >= FECHA_EVENTO).astype(int)
base["grupo"] = np.where(base["tratado"].eq(1), "UK · tratado", "Europa · comparación")
base["semana"] = FECHA_EVENTO + pd.to_timedelta(
    np.floor((base["InvoiceDate"] - FECHA_EVENTO).dt.total_seconds() / (7*24*3600)).astype(int)*7, unit="D")

print("Ventana:", inicio.date(), "a", fin.date())
display(pd.crosstab(base["grupo"], base["post"], values=base["ingreso"], aggfunc=["count","sum"]).round(2))


Control disponible: ['France', 'Germany', 'Netherlands', 'Belgium', 'Spain', 'EIRE']
Ventana: 2010-07-06 a 2011-07-05


count                 sum            
post                       0       1           0           1
grupo                                                       
Europa · comparación   15784   12667   580942.20   461614.54
UK · tratado          276008  184663  5280621.61  3367363.44

### Pregunta 2 — Sesgo de selección
Antes de mirar el DiD, explique por qué UK y los países de comparación podrían diferir estructuralmente. Mencione al menos tres factores.

**Respuesta:**

UK y los países de comparación podrían diferir estructuralmente en factores como el **tamaño y volumen de ventas**, la **composición de productos y clientes**, y las **condiciones o dinámicas propias de cada mercado**. Estas diferencias pueden hacer que sus niveles y tendencias de ingresos sean distintos incluso antes del evento. Por ello, no se debe asumir automáticamente que los países de comparación representan el contrafactual de UK; es necesario **auditar sus tendencias previas, cobertura y composición** antes de interpretar el efecto causal.

## 3. DAG — pensar antes de calcular

Construya en papel o en una celda Markdown un DAG que incluya como mínimo:

**Cambio VAT → precios/decisiones → ventas**  
**País/mercado → composición de clientes → ventas**  
**Estacionalidad → demanda → ventas**  
**Composición de productos → ventas**

Señale qué variables no están observadas en UCI y podrían amenazar la identificación.


### Pregunta 3 — DAG
¿Qué puertas traseras o confusores identifica? ¿Qué variables necesitaría observar para fortalecer el diseño?

**Respuesta:**

Se identifican como posibles confusores el **país o mercado**, la **composición de clientes**, la **estacionalidad de la demanda** y la **composición de productos**, porque pueden afectar las ventas y también diferir entre UK y los mercados de comparación. Para fortalecer el diseño sería necesario observar variables como **precios**, **características y composición de clientes**, **productos vendidos**, **demanda estacional** y otros factores que puedan cambiar simultáneamente con el evento. Estas variables permitirían evaluar mejor si las diferencias observadas se relacionan con el cambio de VAT o con otros factores.

In [ ]:
# CELDA 7 — Agregación semanal
weekly = (base.groupby(["semana","grupo","tratado","post"], as_index=False)
              .agg(ingreso=("ingreso","sum"),
                   unidades=("Quantity","sum"),
                   transacciones=("InvoiceNo","nunique"),
                   clientes=("CustomerID","nunique")))

display(weekly.head())
print("Semanas por grupo:")
display(weekly.groupby(["grupo","post"])["semana"].nunique())


,semana,grupo,tratado,post,ingreso,unidades,transacciones,clientes
0,2010-07-06,Europa · comparación,0,0,31835.53,15283,22,14
1,2010-07-06,UK · tratado,1,0,108062.71,52604,288,215
2,2010-07-13,Europa · comparación,0,0,16764.89,10527,21,11
3,2010-07-13,UK · tratado,1,0,146302.72,78669,336,259
4,2010-07-20,Europa · comparación,0,0,8586.62,4930,21,18


Semanas por grupo:


grupo                 post
Europa · comparación  0       25
                      1       26
UK · tratado          0       25
                      1       26
Name: semana, dtype: int64

In [ ]:
# CELDA 8 — Tendencias observadas
fig = px.line(weekly, x="semana", y="ingreso", color="grupo", markers=True,
              title="Ingreso semanal observado — tratado vs. comparación",
              labels={"semana":"Semana","ingreso":"Ingreso (£)","grupo":"Grupo"})
fig.add_vline(x=FECHA_EVENTO.timestamp()*1000, line_dash="dash")
fig.update_layout(height=470, hovermode="x unified")
fig.show()


## 4. Tendencias paralelas: evaluación previa

No basta con que dos líneas “se parezcan”. Compare la evolución **antes** del evento. Como diagnóstico pedagógico, estimaremos una pendiente lineal pre-evento para cada grupo y compararemos su dirección y magnitud.


In [ ]:
# CELDA 9 — Diagnóstico de pendientes pre
pre = weekly.loc[weekly["post"].eq(0)].copy()
t0 = pre["semana"].min()
pre["t"] = (pre["semana"] - t0).dt.days / 7

pendientes = {}
for g, d in pre.groupby("grupo"):
    if len(d) >= 3:
        b = np.polyfit(d["t"], d["ingreso"], 1)[0]
        pendientes[g] = b

print("Pendientes pre-evento (£ por semana):")
for g,b in pendientes.items():
    print(f"{g:22s}: {b:,.2f}")

if len(pendientes)==2:
    vals=list(pendientes.values())
    print("Diferencia absoluta de pendientes:", f"{abs(vals[0]-vals[1]):,.2f}")


Pendientes pre-evento (£ por semana):
Europa · comparación  : 189.12
UK · tratado          : 8,721.44
Diferencia absoluta de pendientes: 8,532.32


### Pregunta 4 — Tendencias paralelas
¿El supuesto parece razonablemente defendible? Use el gráfico, las pendientes y el conocimiento del caso. Evite responder solo “sí/no”.

**Respuesta:**

El supuesto de tendencias paralelas **no parece razonablemente defendible** con este diagnóstico. Antes del evento, la pendiente del grupo **Europa · comparación** es de aproximadamente **£189,12 por semana**, mientras que la de **UK · tratado** es de **£8.721,44 por semana**, con una diferencia absoluta de **£8.532,32**. Esto muestra que ambos grupos seguían evoluciones muy diferentes antes del cambio de VAT. Por tanto, las tendencias previas no son suficientemente similares para considerar al grupo de comparación como un contrafactual sólido sin realizar análisis adicionales.

In [ ]:
# CELDA 10 — Tabla 2×2 y DiD manual
resumen = (weekly.groupby(["grupo","post"])["ingreso"]
                 .mean().unstack("post"))
resumen.columns = ["Pre","Post"]
resumen["Cambio"] = resumen["Post"] - resumen["Pre"]
display(resumen.round(2))

delta_t = resumen.loc["UK · tratado","Cambio"]
delta_c = resumen.loc["Europa · comparación","Cambio"]
did = delta_t - delta_c

print(f"Δ Tratado     : £ {delta_t:,.2f} por semana")
print(f"Δ Comparación : £ {delta_c:,.2f} por semana")
print(f"DiD           : £ {did:,.2f} por semana")


,Pre,Post,Cambio
grupo,,,
Europa · comparación,23237.69,17754.41,-5483.28
UK · tratado,211224.86,129513.98,-81710.89


Δ Tratado     : £ -81,710.89 por semana
Δ Comparación : £ -5,483.28 por semana
DiD           : £ -76,227.60 por semana


In [ ]:
# CELDA 11 — Regresión DiD
# El coeficiente tratado:post reproduce la lógica DiD sobre el panel semanal agregado.
modelo = smf.ols("ingreso ~ tratado + post + tratado:post", data=weekly).fit(cov_type="HC1")
print(modelo.summary())

beta_did = modelo.params["tratado:post"]
p_did = modelo.pvalues["tratado:post"]
print(f"\nβ_DiD = £ {beta_did:,.2f} por semana")
print(f"p-valor = {p_did:.4f}")


                            OLS Regression Results                            
Dep. Variable:                ingreso   R-squared:                       0.667
Model:                            OLS   Adj. R-squared:                  0.657
Method:                 Least Squares   F-statistic:                     117.0
Date:                Thu, 01 Oct 2026   Prob (F-statistic):           2.88e-32
Time:                        01:58:25   Log-Likelihood:                -1260.9
No. Observations:                 102   AIC:                             2530.
Df Residuals:                      98   BIC:                             2540.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     2.324e+04   1987.748     11.690   

### Pregunta 5 — Interpretación
Interprete β_DiD en **libras por semana**. Después, redacte una segunda frase que deje claro que la interpretación causal está condicionada a los supuestos del diseño.

**Respuesta:**

El coeficiente **β_DiD = −£76.227,60 por semana** indica que, después del evento, el ingreso semanal promedio observado en **UK** disminuyó aproximadamente **£76.227,60 más por semana** que el cambio observado en el grupo de comparación.

Esta interpretación como **efecto causal del aumento del VAT** está condicionada al cumplimiento de los supuestos del diseño, especialmente el de **tendencias paralelas**; como este supuesto no parece razonablemente defendible según el diagnóstico previo, el resultado debe interpretarse con cautela y no como un efecto causal definitivo.

## 5. Robustez

Un resultado causal no debería depender por completo de una sola decisión del analista. Repetiremos el cálculo con:
1. una ventana más corta;
2. un control alternativo.


In [ ]:
# CELDA 12 — Función de robustez
def estimar_did(semanas, controles):
    ini = FECHA_EVENTO - pd.Timedelta(weeks=semanas)
    fin2 = FECHA_EVENTO + pd.Timedelta(weeks=semanas)
    d = ventas.loc[
        ventas["InvoiceDate"].between(ini, fin2, inclusive="left") &
        ventas["Country"].isin(["United Kingdom"] + controles)
    ].copy()
    d["tratado"]=(d["Country"]=="United Kingdom").astype(int)
    d["post"]=(d["InvoiceDate"]>=FECHA_EVENTO).astype(int)
    d["grupo"]=np.where(d["tratado"].eq(1),"UK · tratado","Comparación")
    d["semana"] = FECHA_EVENTO + pd.to_timedelta(
        np.floor((d["InvoiceDate"] - FECHA_EVENTO).dt.total_seconds() / (7*24*3600)).astype(int)*7, unit="D")
    w=(d.groupby(["semana","grupo","tratado","post"],as_index=False)
        .agg(ingreso=("ingreso","sum")))
    m=smf.ols("ingreso ~ tratado + post + tratado:post",data=w).fit(cov_type="HC1")
    return len(w), m.params.get("tratado:post",np.nan), m.pvalues.get("tratado:post",np.nan)

control_a = control_disponible
control_b = [c for c in ["France","Germany","Belgium"] if c in set(ventas["Country"])]

resultados = pd.DataFrame([
    ("Base 26 semanas",)+estimar_did(26,control_a),
    ("Ventana 13 semanas",)+estimar_did(13,control_a),
    ("Control FR-DE-BE",)+estimar_did(26,control_b),
], columns=["Especificación","n_filas_panel","DiD","p_valor"])
display(resultados.round(4))


,Especificación,n_filas_panel,DiD,p_valor
0,Base 26 semanas,102,-76227.6034,0.0010
1,Ventana 13 semanas,50,-146502.6993,0.0001
2,Control FR-DE-BE,102,-80225.7062,0.0005


### Pregunta 6 — Robustez
¿El signo y la magnitud del DiD son estables? Si cambian de manera importante, ¿qué le dice eso sobre la solidez de la conclusión?

**Respuesta:**

Sí, el signo del DiD se mantiene negativo en las tres especificaciones:
- Base, 26 semanas: −£76 227,60 por semana.
- Ventana de 13 semanas: −£146 502,70 por semana.
- Control FR-DE-BE: −£80 225,71 por semana.

Por tanto, la dirección del efecto es estable, pero la magnitud cambia de forma importante, especialmente al reducir la ventana a 13 semanas. Esto indica que el resultado es consistente en su dirección, pero la estimación del tamaño del efecto es sensible a la especificación elegida. Por ello, la conclusión causal debe interpretarse con cautela y mantenerse condicionada a los supuestos del diseño, especialmente a la validez del grupo de comparación y de las tendencias paralelas.

In [ ]:
# CELDA 13 — Peso de cancelaciones como diagnóstico adicional
cancel = df.loc[
    df["InvoiceDate"].between(inicio, fin, inclusive="left") &
    df["Country"].isin(["United Kingdom"] + control_disponible)
].copy()
cancel["es_cancelacion"] = cancel["InvoiceNo"].str.upper().str.startswith("C")
cancel["grupo"] = np.where(cancel["Country"].eq("United Kingdom"),"UK · tratado","Europa · comparación")
display((cancel.groupby("grupo")["es_cancelacion"].agg(["count","mean"])*[1,100]).rename(columns={"mean":"% cancelación"}).round(2))


,count,% cancelación
grupo,,
Europa · comparación,29452,3.36
UK · tratado,471294,1.65


## 6. Conclusión ejecutiva

Complete los cuatro niveles sin exagerar la evidencia:

**1. Evidencia descriptiva:**

Las ventas observadas disminuyeron después del evento en ambos grupos. En Reino Unido, el ingreso semanal promedio pasó de £211 224,86 a £129 513,98; en el grupo de comparación pasó de £23 237,69 a £17 754,41.

**2. Comparación DiD:**

El cambio diferencial estimado fue de −£76 227,60 por semana, lo que indica una caída adicional del ingreso semanal en Reino Unido respecto al grupo de comparación.

**3. Supuestos:**

La evidencia debilita el supuesto de tendencias paralelas, ya que las pendientes pre-evento fueron muy diferentes: £8 721,44 por semana en Reino Unido frente a £189,12 en la comparación. Además, el resultado cambia en magnitud al modificar la ventana o el grupo de control.

**4. Límite causal:**

Con esta base no puede afirmarse que el aumento del VAT haya causado por sí solo la caída observada, porque no se observa directamente el VAT aplicado por transacción ni se garantizan controles no expuestos.

**Recomendación final:**

El análisis muestra una caída diferencial estimada de £76 227,60 por semana en Reino Unido después del evento. Sin embargo, esta cifra no debe interpretarse directamente como un efecto causal del aumento del VAT. El principal problema es que las tendencias previas no parecen suficientemente similares: la pendiente de Reino Unido fue de £8 721,44 por semana, frente a £189,12 en el grupo de comparación. Además, la estimación cambia de −£76 227,60 a −£146 502,70 al utilizar una ventana de 13 semanas. Por ello, la evidencia es compatible con una caída diferencial, pero la identificación causal es débil. La conclusión podría fortalecerse si se dispusiera de un grupo de comparación con tendencias pre-evento más comparables y de información adicional sobre factores que afectan simultáneamente a los mercados, como precios, composición de productos, clientes y demanda. Si esas condiciones se cumplieran, sería posible evaluar con mayor solidez si el cambio estuvo asociado causalmente al aumento del VAT.

## 7. Ticket de salida

**1. ¿Qué cambió en tu conclusión al pasar de una comparación simple a DiD?**

La comparación simple solo muestra que las ventas del Reino Unido disminuyeron después del evento. Con DiD, además, se compara ese cambio con el ocurrido en el grupo de comparación, obteniendo un cambio diferencial estimado de −£76 227,60 por semana.

**2. ¿Cuál es el supuesto más frágil del análisis?**

El supuesto de tendencias paralelas, porque las pendientes pre-evento fueron muy diferentes entre Reino Unido y el grupo de comparación.

**3. Completa:**

“Bajo los supuestos del diseño, la evidencia sugiere que…”
…el ingreso semanal del Reino Unido presentó una disminución diferencial respecto al grupo de comparación después del evento.

**4. Completa:**

“Con estos datos no puedo afirmar que…”
…el aumento del VAT haya causado por sí solo la disminución observada, porque la base no garantiza un grupo de comparación no expuesto ni contiene toda la información necesaria para sostener la identificación causal.

## Referencias

- Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Facure, M. (2023). *Causal Inference in Python*. O’Reilly Media.
- Angrist, J. D., & Pischke, J.-S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
- HM Revenue & Customs. *VAT Notice 700 — Historic VAT rates*. La tasa estándar aumentó a 20 % el 4 de enero de 2011.
